# Stress Prediction — 다차원(Full-Dimensional) 거리 가중 최근접이웃

v34의 pair 방식은 8개 컬럼을 2개씩(28개 조합) 짝지어 각각 1-NN을 찾고 분위수로 합칩니다.
여기서는 **8개 전체를 한 번에 써서 거리를 재고, 가까운 k개를 거리에 반비례한 가중치로 평균**내는
완전히 다른 방식을 테스트합니다. (단순 평균 k-NN은 이미 실패했지만, 가중치를 주는 건 다릅니다.)

트리 모델(tree_prediction)은 v34와 동일하게 유지하고, pair 부분만 이 방식으로 교체해서 블렌드합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


## 다차원 거리 행렬 (한 번만 계산, k/가중치 방식은 사후 탐색)

In [2]:

def full_dim_distance_matrix(train_features, val_features):
    """8개 컬럼을 rank-transform 후 맨해튼 거리 합산 (n_val x n_train)."""
    train_ranks, val_ranks = [], []
    for column in PAIR_COLUMNS:
        tr, va = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
        train_ranks.append(tr)
        val_ranks.append(va)
    train_ranks = np.column_stack(train_ranks)  # (n_train, 8)
    val_ranks = np.column_stack(val_ranks)      # (n_val, 8)
    # 맨해튼 거리: (n_val, n_train)
    distance = np.abs(val_ranks[:, None, :] - train_ranks[None, :, :]).sum(axis=2)
    return distance


def knn_predictions_for_k_grid(distance, target_values, k_grid):
    """미리 계산된 거리 행렬로 여러 k, 두 가지 가중치 방식(uniform/inverse-distance) 예측을 반환."""
    n_val = distance.shape[0]
    max_k = max(k_grid)
    nearest_idx = np.argpartition(distance, kth=max_k - 1, axis=1)[:, :max_k]
    # 각 행에서 실제 거리 기준 정렬 (argpartition은 순서 보장 안 함)
    row_indices = np.arange(n_val)[:, None]
    nearest_dist = distance[row_indices, nearest_idx]
    order = np.argsort(nearest_dist, axis=1)
    nearest_idx = nearest_idx[row_indices, order]
    nearest_dist = nearest_dist[row_indices, order]
    nearest_targets = target_values[nearest_idx]  # (n_val, max_k)

    results = {}
    for k in k_grid:
        idx_k = nearest_targets[:, :k]
        dist_k = nearest_dist[:, :k]
        # uniform 평균
        results[(k, "uniform")] = idx_k.mean(axis=1)
        # inverse-distance 가중 평균 (거리 0인 경우 대비 작은 값 더함)
        inv_w = 1.0 / (dist_k + 1e-3)
        results[(k, "inverse_distance")] = (idx_k * inv_w).sum(axis=1) / inv_w.sum(axis=1)
    return results


## 5-Fold CV — tree는 한 번만 학습, pair는 k/가중치 조합을 전부 사후 평가

In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

K_GRID = [1, 3, 5, 10, 20]
cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

scores = {(k, w): [] for k in K_GRID for w in ("uniform", "inverse_distance")}

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)

    pred_tree = tree_prediction(train_feat, val_feat, y_train)

    distance = full_dim_distance_matrix(train_feat, val_feat)
    pair_preds = knn_predictions_for_k_grid(distance, y_train.to_numpy(dtype=float), K_GRID)

    for (k, w), pair_pred in pair_preds.items():
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pair_pred, 2), 0.0, 1.0)
        scores[(k, w)].append(mean_absolute_error(y_val, combined))

    print(f"Fold {fold} done")

results = pd.DataFrame([
    {"k": k, "weighting": w, "mean_MAE": np.mean(v)}
    for (k, w), v in scores.items()
]).sort_values("mean_MAE").reset_index(drop=True)

print()
print("(참고) v34 원본(2D pair-argmin) CV MAE: 0.148033")
print()
display(results)


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

(참고) v34 원본(2D pair-argmin) CV MAE: 0.148033



,k,weighting,mean_MAE
0,3,inverse_distance,0.162493
1,5,inverse_distance,0.162610
2,1,uniform,0.163317
3,1,inverse_distance,0.163317
4,10,inverse_distance,0.163437
5,20,inverse_distance,0.165047
6,3,uniform,0.174463
7,5,uniform,0.175523
8,10,uniform,0.176107
9,20,uniform,0.176360


## 최적 조합이 v34보다 나으면 최종 제출 파일 생성

위 결과에서 `mean_MAE`가 0.148033보다 **확실히** 낮은 조합이 있을 때만 아래 셀을 실행하세요.
`BEST_K`, `BEST_WEIGHTING`을 결과에 맞게 수정해주세요.


In [4]:

BEST_K = 5              # 위 결과 1위 조합으로 수정
BEST_WEIGHTING = "inverse_distance"  # 위 결과 1위 조합으로 수정

final_tree = tree_prediction(raw_X, raw_X_test, y)
final_distance = full_dim_distance_matrix(raw_X, raw_X_test)
final_pair_preds = knn_predictions_for_k_grid(final_distance, y.to_numpy(dtype=float), [BEST_K])
final_pair = final_pair_preds[(BEST_K, BEST_WEIGHTING)]

final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_fulldim_knn_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_fulldim_knn_0806.csv
Prediction range: 0.01 0.99


,ID,stress_score
0,TEST_0000,0.57
1,TEST_0001,0.96
2,TEST_0002,0.22
3,TEST_0003,0.48
4,TEST_0004,0.51
5,TEST_0005,0.49
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.63
9,TEST_0009,0.81
